# Article 7 — Sensibilidad v0.3

Cargue `Article_7_Sensibilidad_Python_v0.3.zip`. Este paquete conserva el motor v0.2.0 y agrega 24 escenarios de sensibilidad. Ya existen resultados; no es necesario repetirlos para leer el informe. Parámetros ilustrativos, sin calibración empírica.

In [ ]:
from google.colab import files
from pathlib import Path
import tempfile, zipfile, subprocess, sys, json
uploaded=files.upload()
archive=next(x for x in uploaded if x=='Article_7_Sensibilidad_Python_v0.3.zip')
work=Path(tempfile.mkdtemp(prefix='article7_sensitivity_'))
with zipfile.ZipFile(archive) as z: z.extractall(work)
project=work/'Article_7_Sensibilidad_v0.3'
subprocess.run([sys.executable,'-m','pip','install','-r',str(project/'requirements-tested.txt')],check=True)
print(project)

## Verificación y diseño
Las 29 pruebas deben aprobar. La tabla muestra únicamente los cambios de cada escenario respecto al basal.

In [ ]:
subprocess.run([sys.executable,'-m','unittest','discover','-s','tests','-v'],cwd=project,check=True)
import pandas as pd
design=json.loads((project/'configs/sensitivity_design_v0.3.json').read_text())
display(pd.DataFrame([{k:s[k] for k in ['scenario','family','changes']} for s in design['scenarios']]))

## Reproducir las 16.800 simulaciones nuevas
Cambie `RUN_ALL` a True. El proceso puede tardar varios minutos y utiliza dos procesos para limitar recursos en Colab. No combine salidas parciales ni cambie parámetros mientras se ejecuta.

In [ ]:
RUN_ALL=False
if RUN_ALL:
    subprocess.run([sys.executable,'-m','article7.sensitivity','--baseline','reference_baseline','--out','outputs/sensitivity_v0.3','--workers','2'],cwd=project,check=True)
    subprocess.run([sys.executable,'-m','article7.analyze_sensitivity','--source','outputs/sensitivity_v0.3','--baseline','reference_baseline','--out','outputs/sensitivity_analysis_v0.3'],cwd=project,check=True)

## Ver el análisis y descargar
Esta celda se activa al completar la simulación anterior. Para analizar datos ya entregados, consulte las instrucciones del README y extraiga todas las partes de datos en una misma carpeta.

In [ ]:
if RUN_ALL:
    from IPython.display import display, Image
    display(pd.read_csv(project/'outputs/sensitivity_analysis_v0.3/scenario_diagnostics.csv'))
    display(Image(filename=str(project/'outputs/sensitivity_analysis_v0.3/figures/scenario_policy_differences.png')))
    import shutil
    out=shutil.make_archive(str(work/'Article_7_Sensibilidad_reproducida'),'zip',project/'outputs')
    files.download(out)